In [0]:
gold_daily = spark.sql("""
    SELECT
        DATE(transaction_date)                          AS txn_date,
        COUNT(*)                                        AS total_transactions,
        SUM(amount)                                     AS total_amount,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2)     AS fraud_rate_pct,
        ROUND(AVG(amount), 2)                           AS avg_transaction_amount
    FROM silver_transactions
    GROUP BY DATE(transaction_date)
    ORDER BY txn_date
""")
gold_daily.write.format("delta").mode("overwrite").saveAsTable("gold_daily_summary")

gold_city = spark.sql("""
    SELECT
        city,
        COUNT(*)                                        AS total_transactions,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2)     AS fraud_rate_pct,
        ROUND(AVG(amount), 2)                           AS avg_amount
    FROM silver_transactions
    GROUP BY city
    ORDER BY fraud_rate_pct DESC
""")
gold_city.write.format("delta").mode("overwrite").saveAsTable("gold_fraud_by_city")

gold_txn_type = spark.sql("""
    SELECT
        transaction_type,
        COUNT(*)                                        AS total_transactions,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2)     AS fraud_rate_pct
    FROM silver_transactions
    GROUP BY transaction_type
    ORDER BY fraud_rate_pct DESC
""")
gold_txn_type.write.format("delta").mode("overwrite").saveAsTable("gold_fraud_by_type")

gold_customers = spark.sql("""
    SELECT
        customer_id,
        COUNT(*)                                        AS total_transactions,
        SUM(is_fraud)                                   AS fraud_count,
        ROUND(SUM(amount), 2)                           AS total_amount_spent,
        ROUND(AVG(amount), 2)                           AS avg_transaction_amount,
        MAX(is_late_night)                              AS has_late_night_txn
    FROM silver_transactions
    GROUP BY customer_id
    HAVING SUM(is_fraud) > 0
    ORDER BY fraud_count DESC
    LIMIT 50
""")
gold_customers.write.format("delta").mode("overwrite").saveAsTable("gold_high_risk_customers")

print("Gold layer complete!")